# ☀️ Solar Sentinel — YOLO26n Training Notebook

This notebook trains a **YOLO26 Nano** model for solar panel defect detection.

## What we're building
A lightweight object detection model that classifies solar panel conditions into **3 classes**:

| Class ID | Name | Description | Triggers Alert? |
|:---------|:-----|:------------|:----------------|
| 0 | `damage` | Cracks, physical damage, electrical damage | ✅ Yes |
| 1 | `blockage` | Dust, bird droppings, snow, debris | ✅ Yes |
| 2 | `healthy` | Clean, no defects | ❌ No |

## Pipeline
1. Download datasets from Roboflow Universe
2. Merge & remap classes to our 3-class taxonomy
3. Train YOLO26n (transfer learning from COCO pretrained weights)
4. Evaluate on validation set
5. Export to NCNN format for Raspberry Pi 5 deployment

## Hardware Target
- **Raspberry Pi 5** (8GB + Active Cooler + Camera Module 3 Wide)
- Inference via NCNN on ARM Cortex-A76 CPU
- Expected: ~7–12 FPS at 640×640

---
## Cell 1: Environment Setup

Install the required packages:
- **ultralytics** — The YOLO framework (includes YOLO26 support since v8.4.0)
- **roboflow** — SDK to programmatically download datasets from Roboflow Universe

We pin `ultralytics>=8.4.0` because YOLO26 was added in that release (January 2026).

In [ ]:
# Install dependencies
# - ultralytics: the YOLO framework that contains YOLO26
# - roboflow: SDK for downloading annotated datasets from Roboflow Universe
!pip install -q ultralytics>=8.4.0 roboflow

print("✅ Packages installed successfully")

---
## Cell 2: GPU Check

Training a YOLO model requires a GPU. Google Colab provides a free **NVIDIA T4** GPU.

**Before running this cell**, make sure you've enabled GPU:
1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU**
3. Click **Save**

This cell verifies the GPU is available and prints its specifications.

In [ ]:
import torch

# Check if CUDA (GPU) is available
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   CUDA version: {torch.version.cuda}")
    print(f"   PyTorch version: {torch.__version__}")
else:
    print("❌ No GPU detected!")
    print("   Go to Runtime → Change runtime type → T4 GPU")
    print("   Training will be extremely slow on CPU.")

We download **3 datasets** from Roboflow Universe that contain solar panel defect images:

| Dataset | Images | Original Classes |
|:--------|:-------|:-----------------|
| Solar Panel Defect Detection | 773 | Bird Drop, Defective, Dusty, Electrical Damage, Non Defective, Physical Damage, Snow |
| Solar Panel Defect 2 (v4) | 1,189 | Bird-drop panel, Clean panel, Damage panel, Dusty panel, Snow-Covered panel |
| solar panel (v3) | 2,280 | crack, normal, cover, dust |

**Total: ~4,242 images**.


In [ ]:
import os
from pathlib import Path

# ============================================================
# CONFIGURATION — Set your Roboflow API key here
# Get a free key at: https://app.roboflow.com/settings/api
# ============================================================
ROBOFLOW_API_KEY = ""  # <-- Paste your API key here

# Base directory for all datasets
BASE_DIR = Path("/content/datasets")
BASE_DIR.mkdir(exist_ok=True)

# Dataset definitions: (workspace, project, version)
# These are the 3 best RGB solar panel defect datasets on Roboflow
DATASETS = [
    {
        "name": "dataset_1",
        "workspace": "solar-panel-defect-detection",
        "project": "solar-panel-defect-detection-hpser",
        "version": 1,
    },
    {
        "name": "dataset_2",
        "workspace": "solar-panel-defect-ufk3b",
        "project": "solar-panel-defect-2-aeuek",
        "version": 4,
    },
    {
        "name": "dataset_3",
        "workspace": "gao-shou-zheng-b6xqc",
        "project": "solar-panel-0swal",
        "version": 3,
    },
]

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    for ds in DATASETS:
        print(f"\n📥 Downloading {ds['name']}...")
        project = rf.workspace(ds["workspace"]).project(ds["project"])
        dataset = project.version(ds["version"]).download(
            "yolov8",
            location=str(BASE_DIR / ds["name"]),
        )
        print(f"   ✅ Saved to {BASE_DIR / ds['name']}")

    print("\n✅ All datasets downloaded!")
else:
    print("⚠️  No Roboflow API key set.")
    print("   Option 1: Set ROBOFLOW_API_KEY above and re-run")
    print("   Option 2: Manually download from Roboflow Universe:")
    print("     • https://universe.roboflow.com/solar-panel-defect-detection/solar-panel-defect-detection-hpser")
    print("     • https://universe.roboflow.com/internship-project-rvpro/solar-panel-defect-btmyj")
    print("     • https://universe.roboflow.com/solar-panel-defects-ax0gw/solar-panel-defect-w1dfi")
    print("   Upload zips to /content/datasets/ and extract them.")


---
## Cell 4: Class Remapping — The Core Step

This is the most important data preparation step. Each dataset uses **different class names**
for essentially the same defects. We need to unify them into our **3-class taxonomy**:

### Remapping Table

| Original Label | → New Class | Class ID |
|:---------------|:------------|:---------|
| Defective, Physical-Damage, Physical Damage, Electrical-Damage, Electrical Damage, Damage, Damage panel | → `damage` | 0 |
| Bird-drop, Bird Drop, Bird-drop panel, Dusty, Dusty panel, Snow, Snow-Covered, Snow-Covered panel | → `blockage` | 1 |
| Non-Defective, Non Defective, Clean, Clean panel | → `healthy` | 2 |

### How YOLO annotations work
Each image has a `.txt` file with one line per detected object:
```
<class_id> <x_center> <y_center> <width> <height>
```
All coordinates are normalized (0-1). We just need to change `class_id` values.

This cell:
1. Reads each dataset's `data.yaml` to get the original class→ID mapping
2. Creates a remapping from old IDs → new 3-class IDs
3. Rewrites all annotation `.txt` files with the new IDs
4. Merges all datasets into a single unified `solar-sentinel/` directory

In [ ]:
import shutil
import yaml
import hashlib
from pathlib import Path
from collections import Counter

# ============================================================
# CLASS REMAPPING RULES
# Maps every possible original class name → our 3-class system
# ============================================================
CLASS_REMAP = {
    # --- DAMAGE (class 0) ---
    # Physical damage: cracks, broken glass, deformation
    "defective": 0,
    "physical-damage": 0,
    "physical damage": 0,
    "physical_damage": 0,
    "crack": 0,
    # Electrical damage: burnt cells, hotspots, wiring issues
    "electrical-damage": 0,
    "electrical damage": 0,
    "electrical_damage": 0,
    # Generic damage labels
    "damage": 0,
    "damage panel": 0,

    # --- BLOCKAGE (class 1) ---
    # Bird droppings
    "bird-drop": 1,
    "bird drop": 1,
    "bird-drop panel": 1,
    # Dust/dirt accumulation
    "dusty": 1,
    "dusty panel": 1,
    "dust": 1,
    "cover": 1,
    # Snow coverage
    "snow": 1,
    "snow-covered": 1,
    "snow-covered panel": 1,

    # --- HEALTHY (class 2) ---
    "non-defective": 2,
    "non defective": 2,
    "clean": 2,
    "clean panel": 2,
    "normal": 2,
}

# Our target class names (order matters — matches Class IDs above)
TARGET_CLASSES = ["damage", "blockage", "healthy"]

# Output directory for the merged dataset
MERGED_DIR = Path("/content/solar-sentinel")


def read_dataset_yaml(dataset_path: Path) -> dict:
    """Read the data.yaml from a Roboflow-exported dataset.

    This file contains the original class names and their IDs.
    Example content:
        names:
          0: Bird Drop
          1: Defective
          2: Dusty
    """
    yaml_path = dataset_path / "data.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError(f"No data.yaml found at {yaml_path}")

    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    # Roboflow exports class names as {id: name} dict
    return data.get("names", {})


def build_id_remap(original_names: dict) -> dict:
    """Build old_id → new_id mapping for a specific dataset.

    Args:
        original_names: {0: 'Bird Drop', 1: 'Defective', ...}

    Returns:
        {0: 1, 1: 0, 2: 1, ...}  (old_id → new_id)
    """
    remap = {}
    for old_id, name in original_names.items():
        # Normalize: lowercase + strip whitespace for matching
        normalized = name.lower().strip()
        if normalized in CLASS_REMAP:
            remap[int(old_id)] = CLASS_REMAP[normalized]
        else:
            print(f"  ⚠️  Unknown class '{name}' (id={old_id}) — skipping")
    return remap


def remap_annotation_file(txt_path: Path, id_remap: dict) -> list:
    """Rewrite a single YOLO annotation file with remapped class IDs.

    YOLO format: <class_id> <x_center> <y_center> <width> <height>
    We only change the class_id, keeping bounding box coordinates intact.

    Returns list of new class IDs found (for counting).
    """
    lines = txt_path.read_text().strip().splitlines()
    new_lines = []
    class_ids_found = []

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue  # Skip malformed lines

        old_id = int(parts[0])
        if old_id not in id_remap:
            continue  # Skip unknown classes

        new_id = id_remap[old_id]
        # Replace old class ID with new one, keep bbox coordinates
        new_lines.append(f"{new_id} {' '.join(parts[1:])}")
        class_ids_found.append(new_id)

    txt_path.write_text("\n".join(new_lines) + "\n" if new_lines else "")
    return class_ids_found


def merge_datasets():
    """Merge all downloaded datasets into a single unified dataset."""
    # Create output structure
    for split in ["train", "val", "test"]:
        (MERGED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (MERGED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    total_stats = Counter()
    total_images = 0
    seen_hashes = set()  # For deduplication by content hash

    for ds in DATASETS:
        ds_path = BASE_DIR / ds["name"]

        # Find the actual dataset root (Roboflow may nest it)
        # Look for data.yaml to find the root
        yaml_candidates = list(ds_path.rglob("data.yaml"))
        if not yaml_candidates:
            print(f"⚠️  Skipping {ds['name']}: no data.yaml found")
            continue
        ds_root = yaml_candidates[0].parent

        print(f"\n📂 Processing {ds['name']} ({ds_root})")

        # Read original class mapping and build remap
        original_names = read_dataset_yaml(ds_root)
        print(f"   Original classes: {original_names}")
        id_remap = build_id_remap(original_names)
        print(f"   Remap: {id_remap}")

        # Process each split (train/val/test)
        for split in ["train", "valid", "test"]:
            # Roboflow uses 'valid' not 'val'
            out_split = "val" if split == "valid" else split

            img_dir = ds_root / split / "images"
            lbl_dir = ds_root / split / "labels"

            if not img_dir.exists():
                continue

            images = [f for f in img_dir.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
            for img_path in images:
                # Deduplicate identical images by their actual content hash
                file_hash = hashlib.md5(img_path.read_bytes()).hexdigest()
                if file_hash in seen_hashes:
                    continue
                seen_hashes.add(file_hash)

                # Prefix filename with dataset name to avoid collisions
                safe_name = f"{ds['name']}_{img_path.name}"
                lbl_name = f"{ds['name']}_{img_path.stem}.txt"

                # Copy image
                dst_img = MERGED_DIR / "images" / out_split / safe_name
                shutil.copy2(img_path, dst_img)

                # Copy and remap label
                src_lbl = lbl_dir / f"{img_path.stem}.txt"
                dst_lbl = MERGED_DIR / "labels" / out_split / lbl_name

                if src_lbl.exists():
                    shutil.copy2(src_lbl, dst_lbl)
                    ids = remap_annotation_file(dst_lbl, id_remap)
                    for cid in ids:
                        total_stats[TARGET_CLASSES[cid]] += 1
                else:
                    # No annotation = empty file (YOLO interprets as no objects)
                    dst_lbl.write_text("")

                total_images += 1

    # Print summary
    print(f"\n{'='*50}")
    print(f"✅ Merged dataset: {total_images} images")
    print(f"   📁 Location: {MERGED_DIR}")
    print(f"\n   Class distribution (annotations):")
    for cls_name, count in total_stats.most_common():
        print(f"      {cls_name}: {count}")

    # Count per split
    for split in ["train", "val", "test"]:
        n = len(list((MERGED_DIR / "images" / split).glob("*.*")))
        print(f"   {split}: {n} images")


# Run the merge!
merge_datasets()


---
## Cell 5: Dataset YAML Configuration

YOLO requires a `data.yaml` file that tells it:
- Where to find images and labels
- How many classes there are
- What the class names are

This cell generates that file for our merged 3-class dataset.

In [ ]:
import yaml
from pathlib import Path

# Dataset configuration for YOLO26n training
# This tells ultralytics where the data is and what classes to detect
dataset_config = {
    "path": str(MERGED_DIR),          # Root directory of the dataset
    "train": "images/train",          # Training images (relative to path)
    "val": "images/val",              # Validation images
    "test": "images/test",            # Test images (optional)
    "nc": 3,                          # Number of classes
    "names": ["damage", "blockage", "healthy"],  # Class names (order = class ID)
}

# Write the YAML file
yaml_path = MERGED_DIR / "solar_sentinel.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

# Display the generated config
print("📄 Generated dataset config:\n")
print(yaml_path.read_text())
print(f"\n✅ Saved to: {yaml_path}")

---
## Cell 6: Data Exploration

Before training, let's visualize:
1. **Class distribution** — Are the classes balanced? Imbalanced data can cause the model to favor the majority class.
2. **Sample images** — What do the training images look like?

Understanding your data is critical. If one class has 10× more samples than another,
the model may learn to always predict the majority class.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter
from pathlib import Path
import random

TARGET_CLASSES = ["damage", "blockage", "healthy"]
MERGED_DIR = Path("/content/solar-sentinel")

# ---- Class Distribution ----
class_counts = Counter()
label_dir = MERGED_DIR / "labels" / "train"

for txt_file in label_dir.glob("*.txt"):
    for line in txt_file.read_text().strip().splitlines():
        parts = line.strip().split()
        if parts:
            class_counts[int(parts[0])] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#e74c3c", "#f39c12", "#2ecc71"]
counts = [class_counts.get(i, 0) for i in range(3)]

ax1.bar(TARGET_CLASSES, counts, color=colors, edgecolor="white", linewidth=1.5)
ax1.set_title("Training Set — Class Distribution", fontsize=14, fontweight="bold")
ax1.set_ylabel("Number of Annotations")
for i, (name, count) in enumerate(zip(TARGET_CLASSES, counts)):
    ax1.text(i, count + max(counts)*0.02, str(count), ha="center", fontweight="bold")

ax2.pie(counts, labels=TARGET_CLASSES, colors=colors, autopct="%1.1f%%",
        startangle=90, textprops={"fontsize": 12})
ax2.set_title("Class Proportions", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("/content/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTotal annotations: {sum(counts)}")
for name, count in zip(TARGET_CLASSES, counts):
    pct = count / sum(counts) * 100 if sum(counts) > 0 else 0
    print(f"  {name}: {count} ({pct:.1f}%)")

# ---- Sample Images with Bounding Boxes ----
print("\n🖼️  Sample training images with annotations:")
img_dir = MERGED_DIR / "images" / "train"
all_images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
samples = random.sample(all_images, min(6, len(all_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    img = Image.open(img_path)
    ax.imshow(img)
    lbl_path = MERGED_DIR / "labels" / "train" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        w, h = img.size
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                rect = patches.Rectangle(
                    (x1, y1), bw*w, bh*h,
                    linewidth=2, edgecolor=colors[cls_id], facecolor="none"
                )
                ax.add_patch(rect)
                ax.text(x1, y1-5, TARGET_CLASSES[cls_id],
                       color="white", fontsize=9, fontweight="bold",
                       bbox=dict(boxstyle="round,pad=0.2",
                                facecolor=colors[cls_id], alpha=0.8))
    ax.set_title(img_path.name[:30], fontsize=9)
    ax.axis("off")

plt.suptitle("Sample Training Images with Annotations", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Cell 7: Training 🚀

Now we train YOLO26n on our merged dataset. Here's what happens:

### Transfer Learning
We start from `yolo26n.pt` — a model **pre-trained on COCO** (80 object classes, 118K images).
This gives us a massive head start: the model already understands edges, textures, shapes,
and objects. We just fine-tune it to learn our 3 specific solar panel classes.

### Why only 50 epochs?
With transfer learning, the model doesn't learn from scratch — it already knows how to
detect objects. It just needs to adapt to our 3 solar panel classes. This typically
converges in **30–40 epochs**. The `patience=20` early stopping will terminate training
automatically if the model plateaus, so we won't waste GPU time.

### Key Parameters
| Parameter | Value | Why |
|:----------|:------|:----|
| `model` | `yolo26n.pt` | Nano variant — smallest, fastest, perfect for Pi 5 |
| `epochs` | 50 | Converges in ~30–40 with transfer learning; early stopping catches plateau |
| `imgsz` | 640 | Standard YOLO input size — good accuracy |
| `batch` | 16 | Fits in T4 GPU memory (16GB) |
| `patience` | 20 | Stop early if no improvement for 20 epochs (saves time) |
| `optimizer` | auto | YOLO26 defaults to MuSGD (its built-in optimizer) |

### What to watch during training
- **box_loss** — How well the model predicts bounding box coordinates (lower = better)
- **cls_loss** — How well it classifies objects (lower = better)
- **mAP50** — Mean Average Precision at 50% IoU (higher = better, target > 0.5)
- **mAP50-95** — Stricter mAP averaged across IoU thresholds (higher = better)

Training on a T4 GPU should take approximately **10–20 minutes**.

In [ ]:
from ultralytics import YOLO

# Load the pre-trained YOLO26 Nano model
# This downloads ~6MB of COCO-pretrained weights on first run
model = YOLO("yolo26n.pt")

# Train on our solar panel dataset
results = model.train(
    data=str(MERGED_DIR / "solar_sentinel.yaml"),  # Our dataset config
    epochs=100,        # Increased from 50 to 100 for better convergence
    imgsz=640,         # Input image size (matches our Pi 5 inference size)
    batch=16,          # Images per training step (limited by GPU memory)
    patience=20,       # Early stopping: stop if no improvement for 20 epochs
    project="/content/runs",  # Where to save results
    name="solar-sentinel",    # Experiment name
    exist_ok=True,     # Overwrite if re-running
    verbose=True,      # Show training progress
    # Data augmentation (already enabled by default, but these are key ones):
    # - Mosaic: combines 4 images into 1 (great for small datasets)
    # - Fliplr: horizontal flip (solar panels are symmetric)
    # - HSV augmentation: varies brightness/contrast
)

print("\n✅ Training complete!")
print(f"   Best model saved at: /content/runs/solar-sentinel/weights/best.pt")

---
## Cell 8: Validation — How Good Is Our Model?

Now we evaluate the trained model on the **validation set** (images it never saw during training).

### Key Metrics
| Metric | What It Measures | Good Value |
|:-------|:-----------------|:-----------|
| **Precision** | Of all detections, how many are correct? | > 0.7 |
| **Recall** | Of all real defects, how many did we find? | > 0.7 |
| **mAP50** | Overall detection quality (50% overlap threshold) | > 0.5 |
| **mAP50-95** | Stricter quality (averaged across overlap thresholds) | > 0.35 |

For our use case, **recall is more important than precision** — we'd rather have a
few false alarms than miss real damage. The CrewAI agents will filter out false positives
during their detailed analysis.

In [ ]:
from ultralytics import YOLO

# Load the best model from training
best_model = YOLO("/content/runs/solar-sentinel/weights/best.pt")

# Run validation on the validation set
metrics = best_model.val(
    data=str(MERGED_DIR / "solar_sentinel.yaml"),
    imgsz=640,
    verbose=True,
)

# Print results in a readable format
print("\n" + "="*60)
print("📊 VALIDATION RESULTS")
print("="*60)
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")
print(f"  mAP@50:     {metrics.box.map50:.4f}")
print(f"  mAP@50-95:  {metrics.box.map:.4f}")
print("\n  Per-class mAP@50:")
for i, cls_name in enumerate(TARGET_CLASSES):
    if i < len(metrics.box.maps):
        print(f"    {cls_name}: {metrics.box.maps[i]:.4f}")
print("="*60)

# Quick assessment
if metrics.box.map50 > 0.5:
    print("\n✅ Model looks good! Proceeding to export.")
elif metrics.box.map50 > 0.3:
    print("\n⚠️  Model is decent but could improve. Consider:")
    print("   - Training for more epochs")
    print("   - Adding more training data")
    print("   - Adjusting augmentation settings")
else:
    print("\n❌ Model performance is low. Try:")
    print("   - Checking annotation quality (Cell 6)")
    print("   - Using a larger model (yolo26s.pt instead of nano)")
    print("   - Increasing augmentation")

---
## Cell 9: Inference Test — See the Model in Action

Let's run the trained model on some test images to visually confirm it's detecting
damage and blockage correctly.

The model draws **bounding boxes** around detected objects with:
- **Class name** (damage / blockage / healthy)
- **Confidence score** (0.0 to 1.0)

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import random

model = YOLO("/content/runs/solar-sentinel/weights/best.pt")

test_dir = MERGED_DIR / "images" / "test"
test_images = list(test_dir.glob("*.*"))
if not test_images:
    test_dir = MERGED_DIR / "images" / "val"
    test_images = list(test_dir.glob("*.*"))

samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    results = model.predict(source=str(img_path), imgsz=640, conf=0.25, verbose=False)
    annotated = results[0].plot()
    ax.imshow(annotated[:, :, ::-1])
    n_detections = len(results[0].boxes)
    class_names = [results[0].names[int(c)] for c in results[0].boxes.cls]
    title = f"{n_detections} detections: {', '.join(class_names) if class_names else 'none'}"
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.suptitle("🔍 Model Predictions on Test Images", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/inference_results.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Inference test complete. Check the images above.")
print("   If detections look correct → proceed to export!")

---
## Cell 10: Export to NCNN (Raspberry Pi 5 Format)

Our Pi 5 uses an **ARM Cortex-A76** CPU (no GPU). For best performance, we export
the model to **NCNN format** — a high-performance neural network framework specifically
optimized for ARM processors.

### Why NCNN?
| Format | Speed on Pi 5 | Why |
|:-------|:-------------|:----|
| PyTorch (.pt) | Slowest | Not optimized for ARM |
| ONNX (.onnx) | Medium | General purpose, decent ARM support |
| **NCNN** | **Fastest** | **Built specifically for ARM CPUs** |

The export produces a folder containing the optimized model files.
We also export a `.pt` (PyTorch) backup in case NCNN has compatibility issues.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("/content/runs/solar-sentinel/weights/best.pt")

# ---- Export 1: NCNN (primary - for Pi 5 deployment) ----
print("📦 Exporting to NCNN format (optimized for ARM/Pi 5)...")
ncnn_path = model.export(format="ncnn", imgsz=640)
print(f"✅ NCNN model exported to: {ncnn_path}")

# ---- Export 2: ONNX (backup) ----
print("\n📦 Exporting to ONNX format (backup)...")
onnx_path = model.export(format="onnx", imgsz=640)
print(f"✅ ONNX model exported to: {onnx_path}")

print("\n📁 Export summary:")
print(f"   PyTorch (best.pt):  /content/runs/solar-sentinel/weights/best.pt")
print(f"   NCNN (folder):      {ncnn_path}")
print(f"   ONNX (backup):      {onnx_path}")
print(f"\n🍓 Copy the NCNN folder to your Pi 5 at: data/models/best_ncnn_model/")

---
## Cell 11: Download Model — Save to Google Drive

Save all model files to your Google Drive so you can:
1. Download them to your laptop
2. Transfer to the Raspberry Pi 5 via SSH/SCP

### Deployment on Pi 5
After downloading, copy the model to your Pi:
```bash
# From your laptop → Pi 5
scp -r best_ncnn_model/ pi@<pi-ip>:~/solar-sentinel/src/data/models/
scp best.pt pi@<pi-ip>:~/solar-sentinel/src/data/models/
```

Then update `.env`:
```env
YOLO_MODEL_PATH=data/models/best_ncnn_model
```

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

drive_dir = Path("/content/drive/MyDrive/solar-sentinel-models")
drive_dir.mkdir(parents=True, exist_ok=True)

# Copy PyTorch model
pt_src = Path("/content/runs/solar-sentinel/weights/best.pt")
shutil.copy2(pt_src, drive_dir / "best.pt")
print(f"✅ Copied best.pt → {drive_dir / 'best.pt'}")

# Copy NCNN model folder
ncnn_src = Path("/content/runs/solar-sentinel/weights/best_ncnn_model")
ncnn_dst = drive_dir / "best_ncnn_model"
if ncnn_src.exists():
    if ncnn_dst.exists():
        shutil.rmtree(ncnn_dst)
    shutil.copytree(ncnn_src, ncnn_dst)
    print(f"✅ Copied NCNN model → {ncnn_dst}")

# Copy ONNX model
onnx_src = Path("/content/runs/solar-sentinel/weights/best.onnx")
if onnx_src.exists():
    shutil.copy2(onnx_src, drive_dir / "best.onnx")
    print(f"✅ Copied best.onnx → {drive_dir / 'best.onnx'}")

# Copy training plots
results_src = Path("/content/runs/solar-sentinel")
for f in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    src = results_src / f
    if src.exists():
        shutil.copy2(src, drive_dir / f)

print(f"\n{'='*60}")
print(f"🏁 ALL DONE!")
print(f"{'='*60}")
print(f"\nModels saved to Google Drive:")
print(f"  📂 {drive_dir}")
print(f"    ├── best.pt          (PyTorch — for further training)")
print(f"    ├── best_ncnn_model/ (NCNN — for Pi 5 deployment)")
print(f"    ├── best.onnx        (ONNX — backup)")
print(f"    └── *.png            (Training plots)")
print(f"\n🍓 Next: Copy the NCNN model to your Pi 5!")
print(f"   scp -r '{drive_dir}/best_ncnn_model' pi@<pi-ip>:~/solar-sentinel/src/data/models/")

---
## Cell 12: Visualizing Model Quality

Let's check the confusion matrix and F1-confidence curve generated during training.
These plots help understand if the model is confusing specific classes (like 'damage' vs 'blockage')
and at what confidence threshold we achieve the best F1 score.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path("/content/runs/solar-sentinel")

print("📊 Confusion Matrix (True vs Predicted Classes):")
cm_path = run_dir / "confusion_matrix.png"
if cm_path.exists():
    display(Image(filename=str(cm_path), width=800))
else:
    print("   Confusion matrix not found.")

print("\n📈 F1-Confidence Curve (Optimal Confidence Threshold):")
f1_path = run_dir / "F1_curve.png"
if f1_path.exists():
    display(Image(filename=str(f1_path), width=800))
else:
    print("   F1 curve not found.")